# 13. 최종 논문 프로토타입: Confidence-Gated Multimodal QA

앞 단계의 ASR, OCR, 환경음, 데이터 품질, ablation을 하나로 묶습니다.
이 노트북의 결과표와 그림은 논문 `Method`, `Experiments`, `Results`의 뼈대가 됩니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
rng = np.random.default_rng(7)
dataset = pd.DataFrame({
    "sample_id": range(120),
    "ocr_quality": rng.uniform(0.35, 0.98, 120),
    "asr_confidence": rng.uniform(0.40, 0.99, 120),
    "sound_risk": rng.choice([0, 1], size=120, p=[0.82, 0.18]),
    "layout_match": rng.choice([0, 1], size=120, p=[0.30, 0.70]),
})
dataset["difficulty"] = 1 - (0.45 * dataset.ocr_quality + 0.35 * dataset.asr_confidence + 0.20 * dataset.layout_match)

def system_predict(row, system):
    if system == "ocr_only":
        p = 0.40 + 0.45 * row.ocr_quality
        unsafe = row.sound_risk and rng.random() < 0.45
    elif system == "asr_ocr":
        p = 0.35 + 0.30 * row.ocr_quality + 0.25 * row.asr_confidence + 0.10 * row.layout_match
        unsafe = row.sound_risk and rng.random() < 0.32
    elif system == "gated_fusion":
        gate = min(row.ocr_quality, row.asr_confidence)
        p = 0.30 + 0.30 * row.ocr_quality + 0.25 * row.asr_confidence + 0.12 * row.layout_match + 0.10 * gate
        unsafe = row.sound_risk and rng.random() < 0.08
    else:
        raise ValueError(system)
    return rng.random() < min(0.95, p), bool(unsafe)

rows = []
for _, row in dataset.iterrows():
    for system in ["ocr_only", "asr_ocr", "gated_fusion"]:
        correct, unsafe = system_predict(row, system)
        rows.append({"sample_id": row.sample_id, "system": system, "correct": correct, "unsafe": unsafe})
out = pd.DataFrame(rows)
metrics = out.groupby("system").agg(
    qa_accuracy=("correct", "mean"),
    unsafe_action_rate=("unsafe", "mean"),
).reset_index()
metrics["paper_score"] = metrics.qa_accuracy - 2 * metrics.unsafe_action_rate
metrics.to_csv(ARTIFACTS / "final_prototype_metrics.csv", index=False, encoding="utf-8-sig")
metrics.sort_values("paper_score", ascending=False)


In [ ]:
method = '''
Proposed method.
Given ASR transcript confidence c_a, OCR quality c_o, layout match score l, and environmental sound risk r,
the system computes an evidence score s = w_a c_a + w_o c_o + w_l l + w_g min(c_a, c_o).
If r indicates a safety-critical event, execution-type answers are converted into confirmation prompts.
Otherwise, the answer is generated from the highest-scoring OCR/layout evidence.
'''
(ARTIFACTS / "method_section_draft.md").write_text(dedent(method).strip() + "\n", encoding="utf-8")
print(ARTIFACTS / "method_section_draft.md")


In [ ]:
if plt:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].bar(metrics["system"], metrics["qa_accuracy"])
    axes[0].set_ylim(0, 1)
    axes[0].set_title("QA accuracy")
    axes[0].tick_params(axis="x", rotation=20)

    axes[1].bar(metrics["system"], metrics["unsafe_action_rate"], color="tomato")
    axes[1].set_ylim(0, max(0.5, metrics["unsafe_action_rate"].max() + 0.05))
    axes[1].set_title("Unsafe action rate")
    axes[1].tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(ARTIFACTS / "final_prototype_metrics.png", dpi=160)


## 논문 기여문 초안

1. 음성 명령, OCR 문서 증거, 환경음 위험 신호를 함께 평가하는 경량 재현 프로토콜을 제안한다.
2. ASR/OCR confidence와 환경음 safety gate를 결합한 해석 가능한 fusion 방법을 제시한다.
3. 노이즈, 흐림, 알람 조건에서 QA 정확도와 unsafe action rate를 동시에 분석한다.
